## 1. Data Integrity Checks

In [2]:
df = pd.read_csv(DATA_PATH)
df['quarter_period'] = pd.PeriodIndex(df['quarter'], freq='Q')
df['quarter_start'] = df['quarter_period'].dt.to_timestamp(how='start')
df = df.sort_values('quarter_period').reset_index(drop=True)

# src.features owns these rate-change transforms for future curated rebuilds.
# Keep this compatibility block so the current checked-in curated CSV can still
# execute before the generated dataset is refreshed. Rate levels are percentages,
# so differences are percentage-point moves rather than proportional log changes.
RATE_CHANGE_LAG_MAP = {
    'cash_rate': [1],
    'unemployment_rate': [1, 2],
}
for source_col, lags in RATE_CHANGE_LAG_MAP.items():
    if source_col not in df:
        continue
    change_col = f'{source_col}_change'
    df[change_col] = df[source_col].diff()
    for lag in lags:
        df[f'{change_col}_lag{lag}'] = df[change_col].shift(lag)

if {'cpi_yoy', 'trimmed_mean_cpi_yoy'}.issubset(df.columns):
    df['headline_trimmed_mean_yoy_gap'] = df['cpi_yoy'] - df['trimmed_mean_cpi_yoy']

df_model = df.drop(columns=['quarter_period', 'quarter_start'])

schema = pd.DataFrame({'column': df_model.columns, 'dtype': df_model.dtypes.astype(str).values})
display(schema)
display(df_model.head())

validation_records = [record.as_dict() for record in validate_curated_dataset(df_model)]
validation_records.append(validate_curated_with_pandera(df_model).as_dict())
validation_summary = pd.DataFrame(validation_records)
display(validation_summary.head(16))

quarters = df['quarter_period']
expected_quarters = pd.period_range(quarters.min(), quarters.max(), freq='Q')
integrity_summary = pd.DataFrame([
    {'check': 'row_count', 'value': len(df)},
    {'check': 'column_count', 'value': len(df_model.columns)},
    {'check': 'start_quarter', 'value': str(quarters.min())},
    {'check': 'end_quarter', 'value': str(quarters.max())},
    {'check': 'duplicate_quarters', 'value': int(df['quarter'].duplicated().sum())},
    {'check': 'missing_quarters', 'value': int(len(expected_quarters.difference(quarters)))},
    {'check': 'is_sorted', 'value': bool(quarters.is_monotonic_increasing)},
])
display(integrity_summary)

,column,dtype
0,quarter,object
1,cpi_qoq,float64
2,cpi_yoy,float64
3,trimmed_mean_cpi_qoq,float64
4,trimmed_mean_cpi_yoy,float64
5,unemployment_rate,float64
6,unemployment_rate_change,float64
7,cash_rate,float64
8,cash_rate_change,float64
9,wage_price_index,float64


,quarter,cpi_qoq,cpi_yoy,trimmed_mean_cpi_qoq,trimmed_mean_cpi_yoy,unemployment_rate,unemployment_rate_change,cash_rate,cash_rate_change,wage_price_index,wpi_growth,producer_price_index,ppi_growth,commodity_price_index,commodity_growth,wti_price,wti_growth,brent_price,brent_growth,aud_usd,aud_usd_change,household_spending,household_spending_growth,inflation_expectations_business,cpi_yoy_lag1,cpi_yoy_lag4,trimmed_mean_cpi_yoy_lag1,trimmed_mean_cpi_yoy_lag4,cash_rate_lag1,cash_rate_lag2,cash_rate_lag4,unemployment_rate_lag1,unemployment_rate_lag2,unemployment_rate_lag4,wpi_growth_lag1,wpi_growth_lag2,ppi_growth_lag1,ppi_growth_lag2,commodity_growth_lag1,wti_growth_lag1,brent_growth_lag1,aud_usd_change_lag1,cash_rate_change_lag1,unemployment_rate_change_lag1,unemployment_rate_change_lag2,household_spending_growth_lag1,inflation_expectations_business_lag1,covid_shock_down_lag0,covid_shock_rebound_lag1,headline_trimmed_mean_yoy_gap
0,1995Q1,1.6,3.9,0.5,2.2,8.765438,NaN,7.5,NaN,NaN,NaN,NaN,NaN,29.996283,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1.7
1,1995Q2,1.3,4.5,0.9,2.5,8.364857,-0.400582,7.5,0.0,NaN,NaN,NaN,NaN,31.011010,3.382842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.39,3.9,NaN,2.2,NaN,7.5,NaN,NaN,8.765438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.02,0,0,2.0
2,1995Q3,1.2,5.1,1.0,2.9,8.370623,0.005766,7.5,0.0,NaN,NaN,NaN,NaN,30.733831,-0.893806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.28,4.5,NaN,2.5,NaN,7.5,7.5,NaN,8.364857,8.765438,NaN,NaN,NaN,NaN,NaN,3.382842,NaN,NaN,NaN,0.0,-0.400582,NaN,NaN,4.39,0,0,2.2
3,1995Q4,0.8,4.9,0.7,3.1,8.390929,0.020306,7.5,0.0,NaN,NaN,NaN,NaN,30.162208,-1.859915,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.58,5.1,NaN,2.9,NaN,7.5,7.5,NaN,8.370623,8.364857,NaN,NaN,NaN,NaN,NaN,-0.893806,NaN,NaN,NaN,0.0,0.005766,-0.400582,NaN,3.28,0,0,1.8
4,1996Q1,0.5,3.8,0.6,3.2,8.400115,0.009187,7.5,0.0,NaN,NaN,NaN,NaN,29.976517,-0.615640,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.26,4.9,3.9,3.1,2.2,7.5,7.5,7.5,8.390929,8.370623,8.765438,NaN,NaN,NaN,NaN,-1.859915,NaN,NaN,NaN,0.0,0.020306,0.005766,NaN,2.58,0,0,0.6


,dataset,status,rows,columns,start_date,end_date,missing_values,duplicate_dates,notes
0,curated_quarterly_macro_features,PASS,124,50,1995Q1,2025Q4,763,0,ok
1,curated_column:cpi_qoq,PASS,124,1,1995Q1,2025Q4,0,0,ok
2,curated_column:cpi_yoy,PASS,124,1,1995Q1,2025Q4,0,0,ok
3,curated_column:trimmed_mean_cpi_qoq,PASS,124,1,1995Q1,2025Q4,0,0,ok
4,curated_column:trimmed_mean_cpi_yoy,PASS,124,1,1995Q1,2025Q4,0,0,ok
5,curated_column:unemployment_rate,PASS,124,1,1995Q1,2025Q4,0,0,ok
6,curated_column:unemployment_rate_change,PASS,123,1,1995Q1,2025Q4,1,0,ok
7,curated_column:cash_rate,PASS,124,1,1995Q1,2025Q4,0,0,ok
8,curated_column:cash_rate_change,PASS,123,1,1995Q1,2025Q4,1,0,ok
9,curated_column:wage_price_index,PASS,114,1,1995Q1,2025Q4,10,0,ok


,check,value
0,row_count,124
1,column_count,50
2,start_quarter,1995Q1
3,end_quarter,2025Q4
4,duplicate_quarters,0
5,missing_quarters,0
6,is_sorted,True
